# mēķis: atrast visas vietas kur šis minēts (takā konkordance līdzīgi)

# definīcijas

In [23]:
class stulbaisGithubSecretsTikParaizsTikDrosh():
    def encode(sl, s: str) -> bytes:
        # Rotate each UTF-8 byte right by 1. Returns bytes (not str) so the
        # payload can't be mangled by text-encoding round-trips.
        result = bytearray()
        for b in s.encode('utf-8'):
            result.append(((b & 1) << 7) | (b >> 1))
        return bytes(result)

    def encode_escaped(sl, s: str) -> str:
        # Returns a Python bytes literal you can paste into source,
        # e.g. b'\x8a\xbc...'
        encoded = sl.encode(s)
        return "b'" + ''.join(f'\\x{b:02x}' for b in encoded) + "'"

    def decode(sl, s) -> str:
        # Accepts bytes (preferred) or a latin-1 str for backwards-compat.
        if isinstance(s, str):
            s = s.encode('latin-1')
        result = bytearray()
        for b in s:
            result.append(((b & 0x80) >> 7) | ((b << 1) & 0xFF))
        return result.decode('utf-8')

    def print_assignment(sl, name: str, s: str) -> None:
        # Prints a ready-to-paste line: NAME = b'\x..\x..'
        print(f"{name} = {sl.encode_escaped(s)}")


gh = stulbaisGithubSecretsTikParaizsTikDrosh()

In [24]:
import pandas as pd
TOKEN_NT_DATA_FU_GITHUB = b'\xb3\xb4\x3a\x34\xba\x31\xaf\x38\xb0\x3a\xaf\x98\x98\x21\x29\x9a\x23\xa2\x23\xa4\x18\xb2\x38\x19\x39\xa0\xbc\x1c\x3a\xac\x24\xb2\x35\xaf\xa8\x9c\x18\x9c\xb3\xa1\xb8\xa2\xa2\xb2\x26\xb9\x19\x2d\x31\x3c\x39\x35\xb3\x21\xac\x26\xa1\x3d\x29\xa9\xb2\x2a\xb8\x34\x1c\x29\x32\xb8\xa1\xa4\xa1\x18\xac\xa8\xa8\x3b\x37\x26\xa6\xaa\xa4\x25\x23\xa6\x24\x9b\xa2\x33\x34\xa5\xb1\x2c\x1b'
TOKEN_NT_DATA = gh.decode(TOKEN_NT_DATA_FU_GITHUB)
TOKEN_OT_DATA_FU_GITHUB = b'\xb3\xb4\x3a\x34\xba\x31\xaf\x38\xb0\x3a\xaf\x98\x98\xa0\xaa\x28\xa2\x9a\x9b\xa8\x18\xbc\x32\x9a\xa7\xb0\xb3\xb3\x33\x19\xa9\xb0\x3d\xaf\xa3\x3c\xb4\xa5\x9b\x39\x1a\x9a\x36\x3a\x99\x9a\x2b\xb4\x2c\xb6\x19\xb6\xa7\x3a\xac\xa0\xa5\xa3\x38\x33\x3b\x18\xb9\x38\xb0\x2c\x39\xbb\xb2\xb9\x99\xa1\xb3\xaa\xa7\xa6\xb4\x9a\x2d\x2c\x23\x24\xa8\xa9\xa5\xb4\x3d\x18\xb2\x25\x3d\x32\x39'
TOKEN_OT_DATA = gh.decode(TOKEN_OT_DATA_FU_GITHUB)
NT_REPO_DATA_PATH = "ewilpeers/new-testament-data"
OT_REPO_DATA_PATH = "grauziitisos/ot-west-len-data"

In [25]:

import requests
import pandas as pd
def download_csv_from_private_repo(repopath, path_in_repo, token, *args, **kwargs):
	url = f"https://api.github.com/repos/{repopath}/contents/{path_in_repo}"
	r = requests.get(url, headers={
		"Authorization": f"token {token}",
		"Accept": "application/vnd.github.v3.raw"
	})
	r.raise_for_status()
	from io import StringIO
	import pandas as pd
	return pd.read_csv(StringIO(r.text), *args, **kwargs)

In [32]:
import requests
import pandas as pd
from io import BytesIO

def download_csv_from_release(repopath, tag, filename, token=None, *args, **kwargs):
    if token:
        # Private repo: go through the API with auth.
        # First, find the asset ID for this filename within the release.
        api_url = f"https://api.github.com/repos/{repopath}/releases/tags/{tag}"
        meta = requests.get(api_url, headers={
            "Authorization": f"token {token}",
            "Accept": "application/vnd.github+json",
        })
        meta.raise_for_status()
        assets = meta.json()["assets"]
        asset = next((a for a in assets if a["name"] == filename), None)
        if asset is None:
            raise FileNotFoundError(
                f"{filename!r} not found in release {tag!r}. "
                f"Available: {[a['name'] for a in assets]}"
            )

        # Now fetch the asset bytes. octet-stream is the magic header.
        blob = requests.get(asset["url"], headers={
            "Authorization": f"token {token}",
            "Accept": "application/octet-stream",
        })
        blob.raise_for_status()
        return pd.read_csv(BytesIO(blob.content), *args, **kwargs)


    url = f"https://github.com/{repopath}/releases/download/{tag}/{filename}"
    return pd.read_csv(url, *args, **kwargs)


In [33]:
l65_df = download_csv_from_private_repo(OT_REPO_DATA_PATH, "RT65_words.csv", TOKEN_OT_DATA, dtype={'strong_num': str})
l24_df = download_csv_from_private_repo(OT_REPO_DATA_PATH, "JTR2024_words.csv", TOKEN_OT_DATA, dtype={'strong_num': str})
hb_df = download_csv_from_release(OT_REPO_DATA_PATH, "data-v1", "biblehub_hb_en_direct.csv", TOKEN_OT_DATA, dtype={'strong_num': str})
strongs_df = download_csv_from_private_repo(OT_REPO_DATA_PATH, "greek-enh-lxx-apo-OT-blb.csv", TOKEN_OT_DATA, dtype={'strong_num': str})
abp_strongs_df = download_csv_from_private_repo(OT_REPO_DATA_PATH, "biblehub_LXX_EXT_el_en_direct.csv", TOKEN_OT_DATA, dtype={'strong_num': str})
l1694_df = download_csv_from_private_repo(OT_REPO_DATA_PATH, 'GL1694_words.csv', TOKEN_OT_DATA)


/tmp/ipykernel_1798/547696060.py:12: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(StringIO(r.text), *args, **kwargs)


In [34]:
nt_strongs_df = download_csv_from_private_repo(NT_REPO_DATA_PATH, "strongs.csv", TOKEN_NT_DATA)
nt_lv65_df = download_csv_from_private_repo(NT_REPO_DATA_PATH, "latvian_full65.csv", TOKEN_NT_DATA)
nt_l24_df = download_csv_from_private_repo(NT_REPO_DATA_PATH, "JTR2024_words.csv", TOKEN_NT_DATA, dtype={'strong_num': str})
nt_l1694_df = download_csv_from_private_repo(NT_REPO_DATA_PATH, "GL1694_words.csv", TOKEN_NT_DATA)


In [38]:
hb_df.head()

,version https://git-lfs.github.com/spec/v1
0,oid sha256:6cab1a615e1c05a524891d37ccabf2385c2...
1,size 141540551


In [28]:
nt_strongs_g = nt_strongs_df.groupby(["book", "chapter", "verse"])
nt_lv_g = nt_lv65_df.groupby(["book", "chapter", "verse"])
nt_l24_g = nt_l24_df.groupby(["book", "chapter", "verse"])
nt_l1694_g = nt_l1694_df.groupby(["book", "chapter", "verse"])

l65_g = l65_df.groupby(["book", "chapter", "verse"])
l24_g = l24_df.groupby(["book", "chapter", "verse"])
hb_g = hb_df.groupby(["book", "chapter", "verse"])
strongs_g = strongs_df.groupby(["book", "chapter", "verse"])
abp_strongs_g = abp_strongs_df.groupby(["book", "chapter", "verse"])
l1694_g = l1694_df.groupby(["book", "chapter", "verse"])

KeyError: 'book'

# darba lauks